# 🚀 The Deployment Starter Kit
## Ship Your PyTorch Models to Production
**ONNX Runtime · Intel OpenVINO · Nvidia TensorRT**

A companion notebook to **Neural Optimization** by [Think Autonomous](https://thinkautonomous.ai)

---

### What this notebook does:
1. Loads a **real** pretrained semantic segmentation model (DeepLabV3)
2. Runs inference on a **real** driving scene image
3. Exports the model to **ONNX** format
4. Runs inference via **ONNX Runtime** and benchmarks it
5. Runs inference via **Intel OpenVINO** and benchmarks it
6. Runs inference via **Nvidia TensorRT** and benchmarks it (GPU required)
7. Compares all frameworks: speed, accuracy, and visual output

This is the **real deployment pipeline** used in production autonomous systems.

---
## 0. Environment Setup

In [ ]:
!pip install torch torchvision onnx onnxsim onnxruntime openvino Pillow matplotlib -q

# TensorRT: only works on machines with Nvidia GPU + CUDA
# On Colab with GPU runtime, uncomment:
# !pip install tensorrt -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
import time
import os
from PIL import Image
import matplotlib.pyplot as plt
import urllib.request

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Download a Real Driving Scene Image

In [ ]:
# Download a real driving scene from the Cityscapes-style dataset
IMG_URL = 'https://raw.githubusercontent.com/pytorch/hub/master/images/deeplab1.png'
IMG_PATH = 'driving_scene.png'

if not os.path.exists(IMG_PATH):
    urllib.request.urlretrieve(IMG_URL, IMG_PATH)
    print(f'Downloaded: {IMG_PATH}')

# Load and display
original_image = Image.open(IMG_PATH).convert('RGB')
print(f'Image size: {original_image.size}')

plt.figure(figsize=(12, 6))
plt.imshow(original_image)
plt.title('Input: Real Driving Scene')
plt.axis('off')
plt.show()

---
## 2. Load a Real Segmentation Model
We use **DeepLabV3 with MobileNetV3-Large** backbone — a model actually used in mobile/edge deployment for autonomous systems.

In [ ]:
# Load pretrained DeepLabV3 with MobileNetV3 backbone
# This is a real segmentation model used in production edge systems
model = torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(
    pretrained=True
)
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DeepLabV3-MobileNetV3-Large')
print(f'Total parameters: {total_params:,}')
print(f'Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)')

In [ ]:
# Preprocessing pipeline — same as production
preprocess = T.Compose([
    T.Resize(520),
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

input_tensor = preprocess(original_image).unsqueeze(0)  # Add batch dim
print(f'Preprocessed input shape: {input_tensor.shape}')
print(f'Input dtype: {input_tensor.dtype}')

---
## 3. PyTorch Inference (Baseline)
Run the model in pure PyTorch and visualize the segmentation output.

In [ ]:
# Cityscapes color palette for visualization
CITYSCAPES_COLORS = np.array([
    [0, 0, 0],       # background
    [128, 64, 128],   # road
    [244, 35, 232],   # sidewalk
    [70, 70, 70],     # building
    [102, 102, 156],  # wall
    [190, 153, 153],  # fence
    [153, 153, 153],  # pole
    [250, 170, 30],   # traffic light
    [220, 220, 0],    # traffic sign
    [107, 142, 35],   # vegetation
    [152, 251, 152],  # terrain
    [70, 130, 180],   # sky
    [220, 20, 60],    # person
    [255, 0, 0],      # rider
    [0, 0, 142],      # car
    [0, 0, 70],       # truck
    [0, 60, 100],     # bus
    [0, 80, 100],     # train
    [0, 0, 230],      # motorcycle
    [119, 11, 32],    # bicycle
    [128, 128, 128],  # other
], dtype=np.uint8)


def visualize_segmentation(prediction, title='Segmentation'):
    """Convert model output to a color segmentation map."""
    if isinstance(prediction, torch.Tensor):
        seg_map = prediction.argmax(dim=1).squeeze().cpu().numpy()
    else:
        seg_map = prediction.argmax(axis=1).squeeze()
    
    # Map class indices to colors
    h, w = seg_map.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(len(CITYSCAPES_COLORS)):
        color_map[seg_map == cls_id] = CITYSCAPES_COLORS[cls_id]
    
    return color_map, seg_map

In [ ]:
# Run PyTorch inference
with torch.no_grad():
    pytorch_output = model(input_tensor)['out']

pytorch_seg, pytorch_classes = visualize_segmentation(pytorch_output)

# Display
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(T.CenterCrop(480)(T.Resize(520)(original_image)))
axes[0].set_title('Input Image')
axes[0].axis('off')
axes[1].imshow(pytorch_seg)
axes[1].set_title('PyTorch Segmentation Output')
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Count unique classes detected
unique_classes = np.unique(pytorch_classes)
print(f'Classes detected: {len(unique_classes)}')
print(f'Output shape: {pytorch_output.shape}')

In [ ]:
def benchmark(run_fn, name, n_warmup=10, n_runs=50):
    """Generic benchmark function."""
    # Warmup
    for _ in range(n_warmup):
        run_fn()
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        run_fn()
        times.append((time.perf_counter() - start) * 1000)
    
    avg = np.mean(times)
    std = np.std(times)
    fps = 1000 / avg
    print(f'{name}: {avg:.1f} ms ± {std:.1f} ms ({fps:.1f} FPS)')
    return avg


# Benchmark PyTorch
pytorch_time = benchmark(
    lambda: model(input_tensor),
    'PyTorch (CPU)'
)

---
## 4. Export to ONNX
ONNX (Open Neural Network Exchange) is the universal format for deploying models across frameworks and hardware. This is the first step in any production deployment pipeline.

In [ ]:
import onnx
import onnxsim

ONNX_PATH = 'deeplabv3_mobilenetv3.onnx'

# DeepLabV3 returns a dict — we need a wrapper to export cleanly
class SegmentationWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        return self.model(x)['out']

wrapped_model = SegmentationWrapper(model)
wrapped_model.eval()

# Export
print('Exporting to ONNX...')
torch.onnx.export(
    wrapped_model,
    input_tensor,
    ONNX_PATH,
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

# Validate
model_onnx = onnx.load(ONNX_PATH)
onnx.checker.check_model(model_onnx)
print('ONNX model validated ✓')

# Simplify — removes redundant operations
print('Simplifying ONNX graph...')
model_simplified, ok = onnxsim.simplify(model_onnx)
if ok:
    onnx.save(model_simplified, ONNX_PATH)
    print('ONNX model simplified ✓')
else:
    print('Simplification skipped (model already optimal)')

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f'\nONNX model saved: {ONNX_PATH} ({size_mb:.1f} MB)')

---
## 5. ONNX Runtime Inference
ONNX Runtime is the most portable deployment option — runs on CPU, GPU, ARM, and practically any hardware.

In [ ]:
import onnxruntime as ort

print(f'ONNX Runtime version: {ort.__version__}')
print(f'Available providers: {ort.get_available_providers()}')

# Create inference session
# In production, you would select the provider based on available hardware
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(ONNX_PATH, providers=providers)

active_provider = session.get_providers()[0]
print(f'Active provider: {active_provider}')

# Get input/output metadata
input_meta = session.get_inputs()[0]
output_meta = session.get_outputs()[0]
print(f'Input: {input_meta.name} {input_meta.shape} ({input_meta.type})')
print(f'Output: {output_meta.name} {output_meta.shape} ({output_meta.type})')

In [ ]:
# Prepare input as numpy (ONNX Runtime expects numpy, not torch tensors)
onnx_input = input_tensor.numpy()

# Run inference
onnx_output = session.run(
    [output_meta.name],
    {input_meta.name: onnx_input}
)[0]

# Visualize
onnx_seg, _ = visualize_segmentation(onnx_output)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(pytorch_seg)
axes[0].set_title('PyTorch Output')
axes[0].axis('off')
axes[1].imshow(onnx_seg)
axes[1].set_title('ONNX Runtime Output')
axes[1].axis('off')
plt.suptitle('Output Comparison: PyTorch vs ONNX Runtime', fontsize=14)
plt.tight_layout()
plt.show()

# Verify numerical consistency
pytorch_np = pytorch_output.numpy()
max_diff = np.max(np.abs(pytorch_np - onnx_output))
print(f'Max numerical difference: {max_diff:.6f}')
print(f'Outputs match: {"✓" if max_diff < 0.001 else "✗ (check precision)"}')

In [ ]:
# Benchmark ONNX Runtime
onnx_time = benchmark(
    lambda: session.run([output_meta.name], {input_meta.name: onnx_input}),
    f'ONNX Runtime ({active_provider})'
)

---
## 6. Intel OpenVINO Inference
OpenVINO is optimized for Intel hardware (CPUs, integrated GPUs, VPUs). It can read ONNX models directly — no separate conversion step needed.

In [ ]:
from openvino.runtime import Core

# Initialize OpenVINO
ie = Core()
print(f'OpenVINO available devices: {ie.available_devices}')

# Read the ONNX model directly (OpenVINO handles conversion internally)
ov_model = ie.read_model(model=ONNX_PATH)

# Compile for CPU (in production, you'd select the target device)
compiled_model = ie.compile_model(model=ov_model, device_name='CPU')

# Get input/output layers
input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

print(f'Input layer: {input_layer.any_name} {input_layer.shape}')
print(f'Output layer: {output_layer.any_name} {output_layer.shape}')

In [ ]:
# Run inference
ov_input = input_tensor.numpy()
ov_output = compiled_model([ov_input])[output_layer]

# Visualize
ov_seg, _ = visualize_segmentation(ov_output)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(pytorch_seg)
axes[0].set_title('PyTorch')
axes[0].axis('off')
axes[1].imshow(onnx_seg)
axes[1].set_title('ONNX Runtime')
axes[1].axis('off')
axes[2].imshow(ov_seg)
axes[2].set_title('OpenVINO')
axes[2].axis('off')
plt.suptitle('All 3 Frameworks — Same Input, Same Output', fontsize=14)
plt.tight_layout()
plt.show()

max_diff_ov = np.max(np.abs(pytorch_np - ov_output))
print(f'PyTorch vs OpenVINO max diff: {max_diff_ov:.6f}')

In [ ]:
# Benchmark OpenVINO
ov_time = benchmark(
    lambda: compiled_model([ov_input])[output_layer],
    'OpenVINO (CPU)'
)

---
## 7. Nvidia TensorRT Inference
TensorRT delivers the fastest inference on Nvidia GPUs. It optimizes the model graph, fuses layers, and calibrates precision (FP16/INT8).

**Requires:** Nvidia GPU + CUDA + TensorRT installed.

On Google Colab: Runtime → Change runtime type → GPU, then `pip install tensorrt`

In [ ]:
trt_available = False
trt_time = None

try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit
    
    trt_available = True
    print(f'TensorRT version: {trt.__version__}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('TensorRT is available ✓')
    
except ImportError:
    print('TensorRT not available on this machine.')
    print('To use TensorRT:')
    print('  1. Use Google Colab with GPU runtime')
    print('  2. pip install tensorrt pycuda')
    print('  3. Or use an Nvidia Docker container')
    print('')
    print('Skipping TensorRT benchmark — see results from other frameworks below.')

In [ ]:
if trt_available:
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    
    def build_trt_engine(onnx_path, fp16=True):
        """Build a TensorRT engine from an ONNX model."""
        builder = trt.Builder(TRT_LOGGER)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser = trt.OnnxParser(network, TRT_LOGGER)
        
        # Parse ONNX
        with open(onnx_path, 'rb') as f:
            if not parser.parse(f.read()):
                for error in range(parser.num_errors):
                    print(f'ONNX parse error: {parser.get_error(error)}')
                return None
        
        # Build engine
        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
        
        if fp16 and builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
            print('FP16 mode enabled ✓')
        
        print('Building TensorRT engine (this may take a few minutes)...')
        engine = builder.build_serialized_network(network, config)
        
        if engine is None:
            print('Failed to build engine')
            return None
        
        # Deserialize
        runtime = trt.Runtime(TRT_LOGGER)
        engine = runtime.deserialize_cuda_engine(engine)
        print(f'TensorRT engine built ✓')
        return engine
    
    engine = build_trt_engine(ONNX_PATH, fp16=True)

In [ ]:
if trt_available and engine is not None:
    
    class TRTInference:
        """TensorRT inference wrapper — production pattern."""
        
        def __init__(self, engine):
            self.context = engine.create_execution_context()
            
            # Allocate device memory
            self.d_input = cuda.mem_alloc(
                int(np.prod(engine.get_tensor_shape('input'))) * 4
            )
            self.d_output = cuda.mem_alloc(
                int(np.prod(engine.get_tensor_shape('output'))) * 4
            )
            self.output_shape = engine.get_tensor_shape('output')
            self.stream = cuda.Stream()
        
        def infer(self, input_data):
            """Run inference on a numpy array."""
            input_data = np.ascontiguousarray(input_data, dtype=np.float32)
            output_data = np.empty(self.output_shape, dtype=np.float32)
            
            # Transfer input to GPU
            cuda.memcpy_htod_async(self.d_input, input_data, self.stream)
            
            # Set tensor addresses
            self.context.set_tensor_address('input', int(self.d_input))
            self.context.set_tensor_address('output', int(self.d_output))
            
            # Execute
            self.context.execute_async_v3(stream_handle=self.stream.handle)
            
            # Transfer output back to CPU
            cuda.memcpy_dtoh_async(output_data, self.d_output, self.stream)
            self.stream.synchronize()
            
            return output_data
    
    # Create inference object
    trt_infer = TRTInference(engine)
    
    # Run inference
    trt_input = input_tensor.numpy()
    trt_output = trt_infer.infer(trt_input)
    
    # Visualize
    trt_seg, _ = visualize_segmentation(trt_output)
    plt.figure(figsize=(8, 6))
    plt.imshow(trt_seg)
    plt.title('TensorRT Output (FP16)')
    plt.axis('off')
    plt.show()
    
    max_diff_trt = np.max(np.abs(pytorch_np - trt_output))
    print(f'PyTorch vs TensorRT max diff: {max_diff_trt:.6f}')
    print(f'(Larger diff expected with FP16 — this is normal)')
    
    # Benchmark
    trt_time = benchmark(
        lambda: trt_infer.infer(trt_input),
        'TensorRT (GPU, FP16)'
    )

---
## 8. Final Results Comparison

In [ ]:
print('=' * 60)
print('  DEPLOYMENT BENCHMARK RESULTS')
print('  Model: DeepLabV3-MobileNetV3-Large (Semantic Segmentation)')
print('=' * 60)
print(f'')
print(f'  {"Framework":<25} {"Time (ms)":<15} {"FPS":<10} {"Speedup":<10}')
print(f'  {"-"*25} {"-"*15} {"-"*10} {"-"*10}')
print(f'  {"PyTorch (baseline)":<25} {pytorch_time:<15.1f} {1000/pytorch_time:<10.1f} {"1.0x":<10}')
print(f'  {"ONNX Runtime":<25} {onnx_time:<15.1f} {1000/onnx_time:<10.1f} {"{:.1f}x".format(pytorch_time/onnx_time):<10}')
print(f'  {"OpenVINO (CPU)":<25} {ov_time:<15.1f} {1000/ov_time:<10.1f} {"{:.1f}x".format(pytorch_time/ov_time):<10}')

if trt_time is not None:
    print(f'  {"TensorRT (GPU, FP16)":<25} {trt_time:<15.1f} {1000/trt_time:<10.1f} {"{:.1f}x".format(pytorch_time/trt_time):<10}')
else:
    print(f'  {"TensorRT":<25} {"N/A (no GPU)":<15}')

print(f'')
print('=' * 60)

In [ ]:
# Visual comparison of all outputs
n_plots = 4 if trt_available else 3
fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 6))

axes[0].imshow(T.CenterCrop(480)(T.Resize(520)(original_image)))
axes[0].set_title('Input Image', fontsize=14)
axes[0].axis('off')

axes[1].imshow(pytorch_seg)
axes[1].set_title(f'PyTorch\n{pytorch_time:.0f}ms', fontsize=14)
axes[1].axis('off')

axes[2].imshow(ov_seg)
axes[2].set_title(f'OpenVINO\n{ov_time:.0f}ms ({pytorch_time/ov_time:.1f}x faster)', fontsize=14)
axes[2].axis('off')

if trt_available:
    axes[3].imshow(trt_seg)
    axes[3].set_title(f'TensorRT FP16\n{trt_time:.0f}ms ({pytorch_time/trt_time:.1f}x faster)', fontsize=14)
    axes[3].axis('off')

plt.suptitle('Same Model, 3 Deployment Frameworks — Real Segmentation Output', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Production Deployment Checklist

You now have a working pipeline. Here is what a production deployment looks like:

| Step | Action | Tool |
|------|--------|------|
| 1 | Train your model | PyTorch |
| 2 | **Optimize** (prune, quantize, distill) | Neural Optimization course |
| 3 | Export to ONNX | `torch.onnx.export()` |
| 4 | Simplify ONNX graph | `onnxsim` |
| 5 | Choose your target hardware | CPU → OpenVINO, GPU → TensorRT |
| 6 | Benchmark all options | This notebook |
| 7 | Validate output consistency | Compare predictions |
| 8 | Deploy | C++ / ROS2 / Docker |

### What is used in real self-driving cars?
- **Autoware** (open-source autonomous driving stack) uses **TensorRT** and **ONNX Runtime** in C++ for production inference
- **Tesla** uses custom TensorRT-based inference on their FSD chip
- **Waymo** uses TensorFlow Serving and custom accelerators

Step 2 is what separates Play Mode from Pro Player Mode. Learn it in **Neural Optimization** at [thinkautonomous.ai](https://thinkautonomous.ai).

---

*The Deployment Starter Kit — Think Autonomous*